Using of Attention mechanism is supposed to better the output of our implementation as the attention provides the relation and implicit context between between a word with other words in the sentence.

In [1]:
import torch
import torch.nn as nn
import random
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, embed_dim)

        self.rnn = nn.LSTM(embed_dim, self.hid_dim, n_layers, dropout=dropout)
        self.dropout =nn.Dropout(dropout)

    def forward(self, src):
        embedded_src = self.dropout(self.embedding(src))

        out, (hidden, cell) = self.rnn(embedded_src)

        return out, hidden, cell

The encoder architecture will remain same.But, the decoder architecture will implement attention. We are implementing the Bahdanau Attention, which uses FFN to determine attention unlike the Luong. The better among the two depends on the context.

In [3]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, enc_hid_dim, hidden_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hidden_dim
        self.n_layers = n_layers
        self.attention = attention


        self.embedding = nn.Embedding(output_dim, embed_dim)

        self.rnn = nn.LSTM(embed_dim + enc_hid_dim, self.hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(self.hid_dim + enc_hid_dim + embed_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, enc_output):
        input = input.unsqueeze(0)

        embedded_input = self.dropout(self.embedding(input))

        a = self.attention(hidden[-1:], enc_output)

        a = a.unsqueeze(1)
        enc_output = enc_output.permute(1, 0, 2)

        weight_context = torch.bmm(a, enc_output)

        weight_context = weight_context.permute(1, 0, 2)

        rnn_input = torch.cat((embedded_input, weight_context), dim = 2)

        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))

        embedded_input = embedded_input.squeeze(0)
        output = output.squeeze(0)
        weight_context = weight_context.squeeze(0)
        preds = self.fc_out(torch.cat((output, weight_context, embedded_input), dim=1))

        return preds, hidden, cell

In [4]:

class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()

        self.attn = nn.Linear(enc_hid_dim + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, enc_outputs):
        batch_size = enc_outputs.shape[1]
        src_len = enc_outputs.shape[0]

        hidden = hidden.squeeze(0).unsqueeze(1).repeat(1, src_len, 1)

        enc_outputs = enc_outputs.permute(1, 0, 2)

        en = torch.tanh(self.attn(torch.cat((hidden, enc_outputs), dim=2)))


        attention = self.v(en).squeeze(2)

        return F.softmax(attention, dim=1)

In [5]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing=0.5):
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        enc_outputs, hidden, cell = self.encoder(src)

        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, enc_outputs)
            outputs[t] = output

            teacher_force = random.random() < teacher_forcing
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1

        return outputs

In [6]:
import pandas as pd
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import numpy as np

DATA_DIR = "../home/"

train_df = pd.read_csv(f"{DATA_DIR}/en_fr_train.csv").dropna().sample(frac=0.1, random_state=42).reset_index(drop=True)
val_df   = pd.read_csv(f"{DATA_DIR}/en_fr_val.csv").dropna().sample(frac=0.1, random_state=42).reset_index(drop=True)
test_df  = pd.read_csv(f"{DATA_DIR}/en_fr_test.csv").dropna().sample(frac=0.1, random_state=42).reset_index(drop=True)




In [7]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

class Vocabulary:
    PAD, SOS, EOS, UNK = 0, 1, 2, 3
    SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]

    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.tok2idx = {t: i for i, t in enumerate(self.SPECIALS)}
        self.idx2tok = {i: t for t, i in self.tok2idx.items()}

    def build(self, sentences):
        counts = Counter(tok for s in sentences for tok in tokenize(s))
        for tok, cnt in counts.items():
            if cnt >= self.min_freq and tok not in self.tok2idx:
                idx = len(self.tok2idx)
                self.tok2idx[tok] = idx
                self.idx2tok[idx] = tok

    def encode(self, text):
        return (
            [self.SOS]
            + [self.tok2idx.get(t, self.UNK) for t in tokenize(text)]
            + [self.EOS]
        )

    def __len__(self):
        return len(self.tok2idx)

src_vocab = Vocabulary(min_freq=2)
trg_vocab = Vocabulary(min_freq=2)

src_vocab.build(train_df["en"].tolist())
trg_vocab.build(train_df["fr"].tolist())


In [8]:
class TranslationDataset(Dataset):
    def __init__(self, df, src_vocab, trg_vocab):
        self.pairs = [
            (src_vocab.encode(row["en"]), trg_vocab.encode(row["fr"]))
            for _, row in df.iterrows()
        ]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


def collate_fn(batch):
    srcs, trgs = zip(*batch)
    src_lens = [len(s) for s in srcs]
    trg_lens = [len(t) for t in trgs]
    max_src = max(src_lens)
    max_trg = max(trg_lens)
    src_pad = torch.zeros(max_src, len(srcs), dtype=torch.long)
    trg_pad = torch.zeros(max_trg, len(trgs), dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, trgs)):
        src_pad[:len(s), i] = torch.tensor(s)
        trg_pad[:len(t), i] = torch.tensor(t)
    return src_pad, trg_pad


BATCH_SIZE = 128

train_loader = DataLoader(
    TranslationDataset(train_df, src_vocab, trg_vocab),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    TranslationDataset(val_df, src_vocab, trg_vocab),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)
test_loader = DataLoader(
    TranslationDataset(test_df, src_vocab, trg_vocab),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

In [9]:

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)

INPUT_DIM  = len(src_vocab)
OUTPUT_DIM = len(trg_vocab)
EMBED_DIM  = 256
HIDDEN_DIM = 512
N_LAYERS   = 2
DROPOUT    = 0.3
N_EPOCHS   = 10
CLIP       = 1.0

encoder = Encoder(INPUT_DIM, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)

attention = Attention(HIDDEN_DIM, HIDDEN_DIM)

decoder = Decoder(
    OUTPUT_DIM,
    EMBED_DIM,
    HIDDEN_DIM,
    HIDDEN_DIM,
    N_LAYERS,
    DROPOUT,
    attention
)
model   = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab.PAD)

Using device: cuda


In [11]:
from tqdm.auto import tqdm
def train_epoch(model, loader, optimizer, criterion, clip, device):
    model.train()
    total_loss = 0
    for src, trg in tqdm(loader):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        output = output[1:].reshape(-1, output.shape[-1])
        target = trg[1:].reshape(-1)
        loss = criterion(output, target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, teacher_forcing=0.0)
            output = output[1:].reshape(-1, output.shape[-1])
            target = trg[1:].reshape(-1)
            total_loss += criterion(output, target).item()
    return total_loss / len(loader)


best_val_loss = float("inf")
for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP, DEVICE)
    val_loss   = evaluate(model, val_loader, criterion, DEVICE)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model_s1.pt")
    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

  0%|          | 0/182 [00:00<?, ?it/s]

Epoch 01 | train 6.2956 | val 6.1513


  0%|          | 0/182 [00:00<?, ?it/s]

Epoch 02 | train 6.0006 | val 6.1160


  0%|          | 0/182 [00:00<?, ?it/s]

...

Epoch 08 | train 4.7132 | val 5.6376


  0%|          | 0/182 [00:00<?, ?it/s]

Epoch 09 | train 4.5756 | val 5.5959


  0%|          | 0/182 [00:00<?, ?it/s]

Epoch 10 | train 4.4353 | val 5.6227


In [12]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def greedy_decode(model, src_tensor, trg_vocab, max_len=50, device="cpu"):
    model.eval()
    with torch.no_grad():
        enc_outputs, hidden, cell = model.encoder(src_tensor.to(device))
        token = torch.tensor([trg_vocab.SOS], device=device)
        tokens = []
        for _ in range(max_len):
            pred, hidden, cell = model.decoder(token, hidden, cell, enc_outputs)
            token = pred.argmax(1)
            if token.item() == trg_vocab.EOS:
                break
            tokens.append(token.item())
    return [trg_vocab.idx2tok.get(t, "<unk>") for t in tokens]


model.load_state_dict(torch.load("best_model_s1.pt", map_location=DEVICE))
model.eval()

smoothie = SmoothingFunction().method1




In [13]:
import torch.nn.functional as F

def beam_search_decode(model, src_tensor, trg_vocab, beam_width=3, max_len=50, device="cpu"):
    model.eval()
    with torch.no_grad():
        encoder_output, hidden, cell = model.encoder(src_tensor.to(device))

        beam = [{
            'score': 0.0,
            'seq': [trg_vocab.SOS],
            'hidden': hidden,
            'cell': cell
        }]

        for _ in range(max_len):
            all_candidates = []
            all_finished = True

            for node in beam:
                seq = node['seq']
                score = node['score']
                h = node['hidden']
                c = node['cell']

                last_token = seq[-1]

                if last_token == trg_vocab.EOS:
                    all_candidates.append(node)
                    continue

                all_finished = False

                token_tensor = torch.tensor([last_token], device=device)
                pred, new_h, new_c = model.decoder(token_tensor, h, c, encoder_output)


                log_prob = F.log_softmax(pred, dim=1).squeeze(0)

                topk_log_prob, topk_indices = torch.topk(log_prob, beam_width)

                for i in range(beam_width):
                    next_token = topk_indices[i].item()
                    next_score = score + topk_log_prob[i].item()

                    candidate = {
                        'score': next_score,
                        'seq': seq + [next_token],
                        'hidden': new_h,
                        'cell': new_c
                    }
                    all_candidates.append(candidate)

            if all_finished:
                break

            ordered = sorted(all_candidates, key=lambda x: x['score'] / len(x['seq']), reverse=True)

            beam = ordered[:beam_width]

    best_sequence = beam[0]['seq']

    final_tokens = []
    for t in best_sequence[1:]:
        if t == trg_vocab.EOS:
            break
        final_tokens.append(t)

    return [trg_vocab.idx2tok.get(t, "<unk>") for t in final_tokens]

references, hypotheses, hypotheses_beam = [], [], []

In [14]:
for _, row in test_df.iterrows():
    src_ids = torch.tensor(src_vocab.encode(row["en"]), dtype=torch.long).unsqueeze(1)
    ref     = tokenize(row["fr"])
    hyp     = greedy_decode(model, src_ids, trg_vocab, device=DEVICE)
    hyp_b   = beam_search_decode(model, src_ids, trg_vocab, beam_width=3, device=DEVICE)
    references.append([ref])
    hypotheses.append(hyp)
    hypotheses_beam.append(hyp_b)
bleu = corpus_bleu(references, hypotheses, smoothing_function=smoothie)
bleu_beam = corpus_bleu(references, hypotheses_beam, smoothing_function=smoothie)
print(f"Test BLEU: {bleu * 100:.2f}")
print(f"  Beam Search Test BLEU: {bleu_beam * 100:.2f}")




Test BLEU: 3.07
  Beam Search Test BLEU: 4.05


In [15]:
for i in range(5):
    row = test_df.iloc[i]
    src_ids = torch.tensor(src_vocab.encode(row["en"]), dtype=torch.long).unsqueeze(1)
    hyp = greedy_decode(model, src_ids, trg_vocab, device=DEVICE)
    hyp_beam = beam_search_decode(model, src_ids, trg_vocab, beam_width=3, device=DEVICE)
    print("SRC : ", row['en'])
    print("REF :", row['fr'])
    print(f"HYP : {' '.join(hyp)}")
    print(f"BEAM  : {' '.join(hyp_beam)}\n")

SRC :  Let's say I want to sort numbers,  as a simple example I've done it with.
REF : Disons que je veux trier des chiffres, comme un exemple simple de ce avec quoi je l'ai fait.
HYP : je veux dire que je veux un un de j ai ai avec un
BEAM  : je veux je veux dire un un de j ai j ai fait avec un

SRC :  And we shouldn't: Our health care system sucks.
REF : On ne devrait pas. Notre système de santé est mauvais.
HYP : nous ne sommes pas de nos vies de nos vies de la de la
BEAM  : nous ne sommes pas de notre système de nos vies

SRC :  There are 50 million twentysomethings in the United States right now.
REF : Il y a actuellement 50 millions de personnes dans la vingtaine aux Etats-Unis.
HYP : il y a des millions de <unk> dans les états unis en <unk> dans les états unis
BEAM  : il y a des millions de <unk> dans les états unis

SRC :  I am learning that it's a genius idea to use a pair of barbecue tongs  to pick up things that you dropped.   I'm learning that nifty trick where you can char

I have tried to implement the model in two methods. One with greedy decode approach and other with beam search approach.